In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from sentiment_rnn import (
    load_and_prepare_data, plot_sequence_lengths, create_model,
    train_model, evaluate_model, save_model, plot_training, save_results
)

os.makedirs("./data/processed", exist_ok=True)
os.makedirs("./models", exist_ok=True)
os.makedirs("./outputs/tables", exist_ok=True)
os.makedirs("./outputs/figures", exist_ok=True)

# Functions for character-level text generation
from text_generation import (
    load_text_corpus, prepare_character_data, build_character_model,
    train_character_model, generate_text, generate_samples, save_five_verses
)


# Different sampling functions
from sampling_strategies import (
    greedy_sampling, temperature_sampling,
    top_k_sampling, top_p_sampling, choose_next_character
)


## 1. Load and prepare the IMDB data

In [ ]:
MAX_WORDS = 10000
MAX_LENGTH = 200

x_train, y_train, x_val, y_val, x_test, y_test, vocab_size, lengths = load_and_prepare_data(
    max_words=MAX_WORDS,
    max_length=MAX_LENGTH
)

print("Training data:", x_train.shape)
print("Validation data:", x_val.shape)
print("Test data:", x_test.shape)
print("Vocabulary size:", vocab_size)

## 2. Review sequence length

In [ ]:
length_summary = plot_sequence_lengths(lengths, MAX_LENGTH)
display(length_summary)

## 3. A2 - Simple RNN

In [ ]:
rnn_model = create_model("rnn", vocab_size, MAX_LENGTH, units=64, dropout=0.5)
rnn_model.summary()

rnn_history, rnn_time = train_model(
    rnn_model, x_train, y_train, x_val, y_val,
    epochs=10, early_stop=False
)

rnn_loss, rnn_accuracy = evaluate_model(rnn_model, x_test, y_test)
print("RNN test accuracy:", rnn_accuracy)
print("RNN test loss:", rnn_loss)
print("Training time:", round(rnn_time, 2), "seconds")

plot_training(rnn_history, "RNN")
save_model(rnn_model, "simple_rnn")

## 4. A3 - LSTM

In [ ]:
lstm_model = create_model("lstm", vocab_size, MAX_LENGTH, units=128, dropout=0.5, layers=2)
lstm_model.summary()

lstm_history, lstm_time = train_model(
    lstm_model, x_train, y_train, x_val, y_val,
    epochs=10, early_stop=True
)

lstm_loss, lstm_accuracy = evaluate_model(lstm_model, x_test, y_test)
print("LSTM test accuracy:", lstm_accuracy)
print("LSTM test loss:", lstm_loss)
print("Training time:", round(lstm_time, 2), "seconds")

plot_training(lstm_history, "LSTM")
save_model(lstm_model, "lstm")

In [ ]:
experiment_results = []

for layers in [1, 2, 3]:
    for units in [64, 128, 256]:
        for dropout in [0.3, 0.5, 0.7]:
            model = create_model(
                "lstm", vocab_size, MAX_LENGTH,
                units=units, dropout=dropout, layers=layers
            )
            print(layers, units, dropout)
            history, seconds = train_model(
                model, x_train, y_train, x_val, y_val,
                epochs=3, early_stop=True
            )
            loss, accuracy = evaluate_model(model, x_test, y_test)

            experiment_results.append({
                "layers": layers,
                "units": units,
                "dropout": dropout,
                "accuracy": accuracy,
                "loss": loss,
                "training_seconds": seconds
            })

experiment_table = pd.DataFrame(experiment_results)
experiment_table.to_csv("./outputs/tables/lstm_experiments.csv", index=False)
display(experiment_table.sort_values("accuracy", ascending=False))

## 5. A4 - Bidirectional LSTM

In [ ]:
bilstm_model = create_model("bilstm", vocab_size, MAX_LENGTH, units=64, dropout=0.5)
bilstm_model.summary()

bilstm_history, bilstm_time = train_model(
    bilstm_model, x_train, y_train, x_val, y_val,
    epochs=10, early_stop=True
)

bilstm_loss, bilstm_accuracy = evaluate_model(bilstm_model, x_test, y_test)
print("BiLSTM test accuracy:", bilstm_accuracy)
print("BiLSTM test loss:", bilstm_loss)
print("Training time:", round(bilstm_time, 2), "seconds")

plot_training(bilstm_history, "BiLSTM")
save_model(bilstm_model, "bilstm")

## 6. A5 - GRU

In [ ]:
gru_model = create_model("gru", vocab_size, MAX_LENGTH, units=128, dropout=0.5)
gru_model.summary()

gru_history, gru_time = train_model(
    gru_model, x_train, y_train, x_val, y_val,
    epochs=10, early_stop=True
)

gru_loss, gru_accuracy = evaluate_model(gru_model, x_test, y_test)
print("GRU test accuracy:", gru_accuracy)
print("GRU test loss:", gru_loss)
print("Training time:", round(gru_time, 2), "seconds")

plot_training(gru_history, "GRU")
save_model(gru_model, "gru")

## 7. Compare all models

In [ ]:
results = [
    {
        "model": "Simple RNN",
        "accuracy": rnn_accuracy,
        "loss": rnn_loss,
        "training_seconds": rnn_time,
        "parameters": rnn_model.count_params()
    },
    {
        "model": "LSTM",
        "accuracy": lstm_accuracy,
        "loss": lstm_loss,
        "training_seconds": lstm_time,
        "parameters": lstm_model.count_params()
    },
    {
        "model": "BiLSTM",
        "accuracy": bilstm_accuracy,
        "loss": bilstm_loss,
        "training_seconds": bilstm_time,
        "parameters": bilstm_model.count_params()
    },
    {
        "model": "GRU",
        "accuracy": gru_accuracy,
        "loss": gru_loss,
        "training_seconds": gru_time,
        "parameters": gru_model.count_params()
    }
]

comparison = save_results(results)
display(comparison)

# PART B - Character-Level LSTM Text Generation

## B1. Character-level data preparation

In [ ]:
# Load the raw IMDB reviews and prepare character sequences
text = load_text_corpus()

SEQ_LENGTH = 100

x_chars, y_chars, char_to_index, index_to_char, chars, char_frequencies = prepare_character_data(
    text, seq_length=SEQ_LENGTH
)

vocab_size = len(chars)

print("Total characters:", len(text))
print("Unique characters:", vocab_size)
print("Input sequences:", x_chars.shape)
print("Target characters:", y_chars.shape)

display(char_frequencies.head(20))

In [ ]:
# Plot the most common characters
plt.figure(figsize=(10, 4))
char_frequencies.head(20).plot(
    kind="bar", x="character", y="count", legend=False
)
plt.title("20 Most Common Characters")
plt.xlabel("Character")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig("./outputs/figures/character_frequency.png")
plt.show()

## B2. Build and train Character-Level LSTM

In [ ]:
char_model = build_character_model(
    vocab_size=vocab_size,
    seq_length=SEQ_LENGTH
)
char_model.summary()

In [ ]:
# Train for 30 epochs. A sample is printed and a checkpoint is saved every 5 epochs.
char_history = train_character_model(
    char_model,
    x_chars,
    y_chars,
    index_to_char,
    seq_length=SEQ_LENGTH,
    epochs=30,
    batch_size=128,
    seed_text="the world is "
)

In [ ]:
# Example: test the sampling functions with sample probabilities
sample_probabilities = [0.50, 0.25, 0.15, 0.10]

print("Greedy:", greedy_sampling(sample_probabilities))
print("Temperature:", temperature_sampling(sample_probabilities, temperature=1.0))
print("Top-k:", top_k_sampling(sample_probabilities, k=3))
print("Top-p:", top_p_sampling(sample_probabilities, p=0.8))

## B3. Text generation and sampling strategies

In [ ]:
# Greedy: always selects the most likely next character
print(generate_text(
    char_model, "the world is ", index_to_char,
    seq_length=SEQ_LENGTH, length=500, strategy="greedy"
))

In [ ]:
# Temperature sampling: lower values are more predictable,
# higher values usually produce more varied text.
for temp in [0.5, 1.0, 1.5]:
    print("\\nTEMPERATURE:", temp)
    print(generate_text(
        char_model, "the world is ", index_to_char,
        seq_length=SEQ_LENGTH, length=500,
        strategy="temperature", temperature=temp
    ))

In [ ]:
# Top-k sampling: choose from the 5 or 10 most likely characters
for k in [5, 10]:
    print("\\nTOP K:", k)
    print(generate_text(
        char_model, "the world is ", index_to_char,
        seq_length=SEQ_LENGTH, length=500,
        strategy="top_k", top_k=k
    ))

In [ ]:
# Top-p (nucleus) sampling
print(generate_text(
    char_model, "the world is ", index_to_char,
    seq_length=SEQ_LENGTH, length=500,
    strategy="top_p", top_p=0.9
))

## B4. Generate creative text and compare results

In [ ]:
# Five different starting phrases, with multiple sampling methods
seed_texts = [
    "the world is ",
    "i remember ",
    "love is ",
    "the night was ",
    "when i was "
]

sample_table = generate_samples(
    char_model, seed_texts, index_to_char,
    seq_length=SEQ_LENGTH
)

display(sample_table[["seed", "strategy", "temperature"]])

In [ ]:
# Display five generated verses and save them in the output folder
five_verses = save_five_verses(
    char_model, index_to_char, seq_length=SEQ_LENGTH
)

for _, row in five_verses.iterrows():
    print("\\nVERSE", row["verse"])
    print(row["text"])

# PART C - Comparative Analysis


## C1. Compare with Logistic Regression baseline

In [ ]:
import time
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sentiment_rnn import clean_text

# Read the same CSV file
raw_data = pd.read_csv("./data/raw/IMDB Dataset.csv")
raw_data = raw_data[["review", "sentiment"]].dropna()

# Convert labels into 0 and 1
raw_data["sentiment"] = raw_data["sentiment"].map({
    "negative": 0,
    "positive": 1
})
raw_data = raw_data.dropna(subset=["sentiment"])

# Use the same 80/20 split pattern
reviews = raw_data["review"].astype(str).apply(clean_text).tolist()
labels = raw_data["sentiment"].astype(int).values

import numpy as np
np.random.seed(42)
indexes = np.random.permutation(len(reviews))
split = int(len(indexes) * 0.8)
train_indexes = indexes[:split]
test_indexes = indexes[split:]

train_reviews = [reviews[i] for i in train_indexes]
test_reviews = [reviews[i] for i in test_indexes]
train_labels = labels[train_indexes]
test_labels = labels[test_indexes]

# Convert reviews into TF-IDF features
tfidf = TfidfVectorizer(max_features=10000)
x_train_tfidf = tfidf.fit_transform(train_reviews)
x_test_tfidf = tfidf.transform(test_reviews)

# Train Logistic Regression
logistic_model = LogisticRegression(max_iter=1000)
start = time.time()
logistic_model.fit(x_train_tfidf, train_labels)
logistic_training_time = time.time() - start

# Measure test accuracy and inference time
start = time.time()
logistic_predictions = logistic_model.predict(x_test_tfidf)
logistic_inference_time = (time.time() - start) / len(test_reviews) * 1000
logistic_accuracy = accuracy_score(test_labels, logistic_predictions)

# Save the baseline model and TF-IDF vectorizer
joblib.dump(logistic_model, "./models/logistic_regression.pkl")
joblib.dump(tfidf, "./models/tfidf_vectorizer.pkl")

print("Logistic Regression accuracy:", logistic_accuracy)
print("Training time:", round(logistic_training_time, 2), "seconds")
print("Inference time per review:", round(logistic_inference_time, 4), "ms")

In [ ]:
def get_inference_time(model, x_test):
    start = time.time()
    model.predict(x_test, batch_size=1, verbose=0)
    total_time = time.time() - start
    return (total_time / len(x_test)) * 1000

rnn_inference = get_inference_time(rnn_model, x_test)
lstm_inference = get_inference_time(lstm_model, x_test)
bilstm_inference = get_inference_time(bilstm_model, x_test)
gru_inference = get_inference_time(gru_model, x_test)

In [ ]:
# Create the comparison table
comparison_all = pd.DataFrame([
    {
        "Model Architecture": "Logistic Regression (TF-IDF)",
        "Accuracy": logistic_accuracy,
        "Training Time (seconds)": logistic_training_time,
        "Parameters": len(logistic_model.coef_.ravel()) + len(logistic_model.intercept_),
        "Inference Time (ms/review)": logistic_inference_time
    },
    {
        "Model Architecture": "Simple RNN 1 layer",
        "Accuracy": rnn_accuracy,
        "Training Time (seconds)": rnn_time,
        "Parameters": rnn_model.count_params(),
        "Inference Time (ms/review)": rnn_inference
    },
    {
        "Model Architecture": "LSTM 2 layers",
        "Accuracy": lstm_accuracy,
        "Training Time (seconds)": lstm_time,
        "Parameters": lstm_model.count_params(),
        "Inference Time (ms/review)": lstm_inference
    },
    {
        "Model Architecture": "Bidirectional LSTM 2 layers",
        "Accuracy": bilstm_accuracy,
        "Training Time (seconds)": bilstm_time,
        "Parameters": bilstm_model.count_params(),
        "Inference Time (ms/review)": bilstm_inference
    },
    {
        "Model Architecture": "GRU 2 layers",
        "Accuracy": gru_accuracy,
        "Training Time (seconds)": gru_time,
        "Parameters": gru_model.count_params(),
        "Inference Time (ms/review)": gru_inference
    }
])

comparison_all.to_csv("./outputs/tables/part_c_model_comparison.csv", index=False)
display(comparison_all)

## C2. Visualize model architectures

In [ ]:
from keras.utils import plot_model

models_to_plot = {
    "simple_rnn": rnn_model,
    "lstm": lstm_model,
    "bilstm": bilstm_model,
    "gru": gru_model
}

for model_name, model in models_to_plot.items():
    try:
        plot_model(
            model,
            to_file="./outputs/figures/" + model_name + "_architecture.png",
            show_shapes=True,
            show_layer_names=True
        )
        print(model_name, "architecture image saved.")
    except Exception as error:
        print("Could not draw", model_name, "architecture.")
        print("Install Graphviz and pydot to enable plot_model.")
        print(error)

In [ ]:
parameter_table = pd.DataFrame([
    {"Model": "Simple RNN", "Parameters": rnn_model.count_params()},
    {"Model": "LSTM", "Parameters": lstm_model.count_params()},
    {"Model": "BiLSTM", "Parameters": bilstm_model.count_params()},
    {"Model": "GRU", "Parameters": gru_model.count_params()}
])

parameter_table.to_csv("./outputs/tables/parameter_comparison.csv", index=False)

plt.figure(figsize=(8, 4))
plt.bar(parameter_table["Model"], parameter_table["Parameters"])
plt.title("Model Parameter Comparison")
plt.xlabel("Model")
plt.ylabel("Number of parameters")
plt.tight_layout()
plt.savefig("./outputs/figures/parameter_comparison.png")
plt.show()

display(parameter_table)